# 03 — Exploratory Data Analysis

Inspects per-class axis statistics, correlations, time-domain window features, and generates EDA figures under `results/plots/`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import load_config
from src.data import clean_sensor_dataframe, filter_and_map_activities, load_processed_splits, load_wisdm_raw
from src.features import (
    compute_axis_correlations,
    compute_class_feature_stats,
    compute_window_time_domain_features,
    plot_eda_summary,
)

cfg = load_config(PROJECT_ROOT / "configs" / "config.yaml")
data_cfg = cfg["data"]

raw_df = load_wisdm_raw(
    raw_dir=data_cfg["raw_dir"],
    devices=data_cfg["devices"],
    sensors=data_cfg["sensors"],
    fallback_dir=PROJECT_ROOT / data_cfg["fallback_raw_dir"],
)
df = filter_and_map_activities(clean_sensor_dataframe(raw_df, data_cfg["features"]), data_cfg["activity_map"])

display(compute_class_feature_stats(df, features=data_cfg["features"]))
display(compute_axis_correlations(df, features=data_cfg["features"]))

saved_plots = plot_eda_summary(df, PROJECT_ROOT / cfg["results"]["plots_dir"], data_cfg["class_names"], data_cfg["features"])
print("Saved plots:", [p.name for p in saved_plots])

In [ ]:
splits = load_processed_splits(PROJECT_ROOT / data_cfg["processed_dir"])
win_feats = compute_window_time_domain_features(splits["X_train"], axis_names=data_cfg["features"])
win_feats.describe().T.head(12)